In [1]:
# import libraries
import pandas as pd
import re

print(pd.__version__)

2.2.2


In [2]:
# load the pokemon dataset
pokemon = pd.read_csv('https://raw.githubusercontent.com/sjsu-cs122-s26/122_datasets/refs/heads/main/Pokemon.csv')
print(pokemon.head())



   #                   Name Type 1  Type 2  Total  HP  Attack  Defense  \
0  1              Bulbasaur  Grass  Poison    318  45      49       49   
1  2                Ivysaur  Grass  Poison    405  60      62       63   
2  3               Venusaur  Grass  Poison    525  80      82       83   
3  3  VenusaurMega Venusaur  Grass  Poison    625  80     100      123   
4  4             Charmander   Fire     NaN    309  39      52       43   

   Sp. Atk  Sp. Def  Speed  Generation  Legendary  
0       65       65     45           1      False  
1       80       80     60           1      False  
2      100      100     80           1      False  
3      122      120     80           1      False  
4       60       50     65           1      False  


In [3]:
# dataset schema / baseline stats
print('Row count:', len(pokemon))
print('Column count:', len(pokemon.columns))
print('Columns:', pokemon.columns.tolist())
print('Unique pokemon names:', pokemon['Name'].nunique())
print('\nGeneration count:\n', pokemon['Generation'].value_counts().sort_index())
print('\nType 1 count (top 10):\n', pokemon['Type 1'].value_counts().head(10))
print('\nType 2 count (top 10):\n', pokemon['Type 2'].value_counts().head(10))
print('\nMissing values per column:\n', pokemon.isna().sum())

# try to

Row count: 800
Column count: 13
Columns: ['#', 'Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
Unique pokemon names: 800

Generation count:
 Generation
1    166
2    106
3    160
4    121
5    165
6     82
Name: count, dtype: int64

Type 1 count (top 10):
 Type 1
Water       112
Normal       98
Grass        70
Bug          69
Psychic      57
Fire         52
Rock         44
Electric     44
Ground       32
Ghost        32
Name: count, dtype: int64

Type 2 count (top 10):
 Type 2
Flying      97
Ground      35
Poison      34
Psychic     33
Fighting    26
Grass       25
Fairy       23
Steel       22
Dark        20
Dragon      18
Name: count, dtype: int64

Missing values per column:
 #               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype

In [4]:
# clean text columns
text_cols = ['Name', 'Type 1', 'Type 2']
for col in text_cols:
    if col in pokemon.columns:
        pokemon[col] = pokemon[col].astype(str).str.strip()

In [5]:
# fill missing secondary type
if 'Type 2' in pokemon.columns:
    pokemon['Type 2'] = pokemon['Type 2'].replace('nan', pd.NA)
    pokemon['Type 2'] = pokemon['Type 2'].fillna('None')

In [6]:
#just to make sure all values are numeric
numeric_cols = ['#', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
for col in numeric_cols:
    if col in pokemon.columns:
        pokemon[col] = pd.to_numeric(pokemon[col], errors='coerce')

In [7]:
# recompute total stats and compare to Total column
stat_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
pokemon['Computed Total'] = pokemon[stat_cols].sum(axis=1)

if 'Total' in pokemon.columns:
    pokemon['Total Match'] = pokemon['Total'] == pokemon['Computed Total']

print(pokemon[['Name', 'Total', 'Computed Total', 'Total Match']].head())

                    Name  Total  Computed Total  Total Match
0              Bulbasaur    318             318         True
1                Ivysaur    405             405         True
2               Venusaur    525             525         True
3  VenusaurMega Venusaur    625             625         True
4             Charmander    309             309         True


In [8]:
# create features for later scoring
pokemon['Offense Score'] = pokemon[['Attack', 'Sp. Atk']].mean(axis=1)
pokemon['Defense Score'] = pokemon[['Defense', 'Sp. Def']].mean(axis=1)
pokemon['Bulk Score'] = pokemon[['HP', 'Defense', 'Sp. Def']].mean(axis=1)
pokemon['Mobility Score'] = pokemon['Speed']
pokemon['Best Attack'] = pokemon[['Attack', 'Sp. Atk']].max(axis=1)
pokemon['Best Defense'] = pokemon[['Defense', 'Sp. Def']].max(axis=1)

In [9]:
# normalize main scoring features for comparison
score_cols = ['Offense Score', 'Defense Score', 'Bulk Score', 'Mobility Score']
for col in score_cols:
    pokemon[col + ' Norm'] = (pokemon[col] - pokemon[col].min()) / (pokemon[col].max() - pokemon[col].min())

In [10]:
# simple viability score for early baseline
pokemon['Viability Score'] = (
    0.35 * pokemon['Offense Score Norm'] +
    0.25 * pokemon['Defense Score Norm'] +
    0.20 * pokemon['Bulk Score Norm'] +
    0.20 * pokemon['Mobility Score Norm']
)

pokemon[['Name', 'Type 1', 'Type 2', 'Viability Score']].sort_values(
    by='Viability Score',
    ascending=False
).head(10)

,Name,Type 1,Type 2,Viability Score
164,MewtwoMega Mewtwo Y,Psychic,None,0.689235
163,MewtwoMega Mewtwo X,Psychic,Fighting,0.688567
426,RayquazaMega Rayquaza,Dragon,Flying,0.687407
422,KyogrePrimal Kyogre,Water,None,0.679316
424,GroudonPrimal Groudon,Ground,Fire,0.679316
796,DiancieMega Diancie,Rock,Fairy,0.635208
418,LatiasMega Latias,Dragon,Psychic,0.621291
413,MetagrossMega Metagross,Steel,Psychic,0.620825
552,Arceus,Normal,None,0.619951
420,LatiosMega Latios,Dragon,Psychic,0.618960


In [11]:
# assign basic battle role
def assign_role(row):
    if row['Speed'] >= 100 and row['Best Attack'] >= 100:
        return 'DPS'
    elif row['HP'] >= 90 and row['Best Defense'] >= 100:
        return 'Tank'
    elif row['Best Attack'] >= 110:
        return 'Heavy hitter'
    elif row['Speed'] >= 110:
        return 'Fast'
    else:
        return 'Balanced'

pokemon['Role'] = pokemon.apply(assign_role, axis=1)
pokemon[['Name', 'Role']].head()

,Name,Role
0,Bulbasaur,Balanced
1,Ivysaur,Balanced
2,Venusaur,Balanced
3,VenusaurMega Venusaur,Heavy hitter
4,Charmander,Balanced


In [12]:
# try to decompose the pokemon into its base and its form
pattern = re.compile(r'[a-z][A-Z0-9%]')

# given the DF, get a set of the base name of the pokemons
def extract_base_name(df):
  seen = set()
  for _, group in df.groupby('#'):
    names = group['Name'].tolist()
    cleaned_names = [name for name in names if not pattern.search(name)]

    # single pokemon name?
    if cleaned_names:
      seen.update(cleaned_names)
    else:
      # the pokemon name is joined with its form
      # take the first one and infer the base
      first = names[0]
      m = pattern.search(first)
      cleaned_name = first[:m.start() + 1] if m else first # GiratinaAltered Forme -> Giratina
      seen.add(cleaned_name)
  return seen

# decompose the pokemon into its name and form
def parse_pokemon_name(name, base_names):
  if name in base_names:
    return name, 'Base'

  # split at the first occurence of an uppercase character
  m = re.search(r'(?<=[a-z])(?=[A-Z0-9%])', name)
  if m:
    idx = m.start()
    return name[:idx], name[idx:]

base_names = extract_base_name(pokemon)
tmp = pokemon['Name'].map(lambda name: parse_pokemon_name(name, base_names))
# add a new base_name column to the df
pokemon.insert(pokemon.columns.get_loc('Name') + 1, 'base_name', tmp.map(lambda x: x[0]))
# add a new form column to the df
pokemon.insert(pokemon.columns.get_loc('base_name') + 1, 'form', tmp.map(lambda x: x[1]))

In [18]:
# https://pokemondb.net/type

a = 0 # no effect
b = 0.5 # not very effective
c = 1 # normal
d = 2.0 # super-effective

# column = defense
# row = attack
effectiveness_matrix = [
  # normal
  [c, c, c, c, c, c, c, c, c, c, c, c, b, a, c, c, b, c],
  # fire
  [c, b, b, c, d, d, c, c, c, c, c, d, b, c, b, c, d, c],
  # water
  [c, d, b, c, b, c, c, c, d, c, c, c, d, c, b, c, c, c],
  # electric
  [c, c, d, b, b, c, c, c, a, d, c, c, c, c, b, c, c, c],
  # grass
  [c, b, d, c, b, c, c, b, d, b, c, b, d, c, b, c, b, c],
  # ice
  [c, b, b, c, d, b, c, c, d, d, c, c, c, c, d, c, b, c],
  # fighting
  [d, c, c, c, c, d, c, b, c, b, b, b, d, a, c, d, d, b],
  # poison
  [c, c, c, c, d, c, c, b, b, c, c, c, b, b, c, c, a, d],
  # ground
  [c, d, c, d, b, c, c, d, c, a, c, b, d, c, c, c, d, c],
  # flying
  [c, c, c, b, d, c, d, c, c, c, c, d, b, c, c, c, b, c],
  # psychic
  [c, c, c, c, c, c, d, d, c, c, b, c, c, c, c, a, b, c],
  # bug
  [c, b, c, c, d, c, b, b, c, b, d, c, c, b, c, d, b, b],
  # rock
  [c, d, c, c, c, d, b, c, b, d, c, d, c, c, c, c, b, c],
  # ghost
  [a, c, c, c, c, c, c, c, c, c, d, c, c, d, c, b, c, c],
  # dragon
  [c, c, c, c, c, c, c, c, c, c, c, c, c, c, d, c, b, a],
  # dark
  [c, c, c, c, c, c, b, c, c, c, d, c, c, d, c, b, c, b],
  # steel
  [c, b, b, b, c, d, c, c, c, c, c, c, d, c, c, c, b, d],
  # fairy
  [c, b, c, c, c, c, d, b, c, c, c, c, c, c, d, d, b, c]
]

for row in effectiveness_matrix:
  assert(len(row) == 18)

# types = {'normal','fire','water','electric','grass','ice','fighting','poison','ground','flying','psychic','bug','rock','ghost','dragon','dark','steel','fairy'}
# types_from_data = set()

# for a in pokemon['Type 1']:
#   types_from_data.add(a.lower())
# for b in pokemon['Type 2']:
#   types_from_data.add(b.lower())

# print('missing :: ', types_from_data - types)


missing ::  {'none'}


In [14]:
  # final cleaned preview
print('Final shape:', pokemon.shape)
print('\nRole counts:\n', pokemon['Role'].value_counts())
# pokemon.head()
pokemon

Final shape: (800, 29)

Role counts:
 Role
Balanced        525
Heavy hitter    106
DPS              95
Tank             55
Fast             19
Name: count, dtype: int64


,#,Name,base_name,form,Type 1,Type 2,Total,HP,Attack,Defense,...,Bulk Score,Mobility Score,Best Attack,Best Defense,Offense Score Norm,Defense Score Norm,Bulk Score Norm,Mobility Score Norm,Viability Score,Role
0,1,Bulbasaur,Bulbasaur,Base,Grass,Poison,318,45,49,49,...,53.000000,45,65,65,0.276471,0.185882,0.217073,0.228571,0.232364,Balanced
1,2,Ivysaur,Ivysaur,Base,Grass,Poison,405,60,62,63,...,67.666667,60,80,80,0.358824,0.254118,0.324390,0.314286,0.316853,Balanced
2,3,Venusaur,Venusaur,Base,Grass,Poison,525,80,82,83,...,87.666667,80,100,100,0.476471,0.348235,0.470732,0.428571,0.433684,Balanced
3,3,VenusaurMega Venusaur,Venusaur,Mega Venusaur,Grass,Poison,625,80,100,123,...,107.666667,80,122,123,0.594118,0.489412,0.617073,0.428571,0.539423,Heavy hitter
4,4,Charmander,Charmander,Base,Fire,None,309,39,52,43,...,44.000000,65,60,50,0.270588,0.136471,0.151220,0.342857,0.227639,Balanced
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,719,Diancie,Diancie,Base,Rock,Fairy,600,50,100,150,...,116.666667,50,100,150,0.529412,0.623529,0.682927,0.257143,0.529190,Balanced
796,719,DiancieMega Diancie,Diancie,Mega Diancie,Rock,Fairy,700,50,160,110,...,90.000000,110,160,110,0.882353,0.435294,0.487805,0.600000,0.635208,DPS
797,720,HoopaHoopa Confined,Hoopa,Hoopa Confined,Psychic,Ghost,600,80,110,60,...,90.000000,70,150,130,0.705882,0.364706,0.487805,0.371429,0.510082,Heavy hitter
798,720,HoopaHoopa Unbound,Hoopa,Hoopa Unbound,Psychic,Dark,680,80,160,60,...,90.000000,80,170,130,0.911765,0.364706,0.487805,0.428571,0.593569,Heavy hitter


In [15]:
# write the new csv file
pokemon.to_csv('/tmp/pokemon_clean.csv', index=False)